# Simlin in Google Colab

Open this notebook in Colab (`File > Open notebook > GitHub`, or the
`colab.research.google.com/github/...` URL of this file), run the cells in
order, and you have a system dynamics model you can edit in the diagram
editor and simulate from Python -- nothing to install beyond `pip`.

Status: pysimlin's editor is verified on JupyterLab 4 by an automated
browser test; on Colab it is expected to work (anywidget supports Colab)
but not yet verified -- if the editor does not appear in the display cell,
run `import os; os.environ["SIMLIN_WIDGET_ASSET"] = "inline"` in a cell
BEFORE the `import simlin` below and try again, and please report which
worked (see `docs/notebook-hosts.md` in the pysimlin source). Everything
below also runs in JupyterLab, Notebook 7, or VS Code.

## Install

pysimlin ships the editor and its engine inside the wheel; `pip` is the whole
install. Colab has no `simlin` preinstalled, so this cell downloads it (a few
seconds).

In [ ]:
%pip install --quiet pysimlin

## Build and open a model

A model file on disk is the source of truth for the editor, so build the
logistic-growth model from the pysimlin README, save it next to this notebook
(`/content` in Colab), and open that file. `simlin.open` keeps the model
attached to the file: edits made in the editor and from Python are written
back to it, and changes made to it by anything else are picked up.

In [ ]:
from pathlib import Path

import simlin
from simlin import Aux, Flow, Stock

model_path = Path.cwd() / "logistic-growth.stmx"

project = simlin.Project.new(
    name="logistic-growth", sim_start=0, sim_stop=100, dt=0.25, time_units="years"
)
with project.get_model().edit() as (_, patch):
    patch.upsert(Stock(name="population", initial_equation="50", inflows=["net_growth"]))
    patch.upsert(Flow(name="net_growth", equation="population * fractional_growth"))
    patch.upsert(
        Aux(
            name="fractional_growth",
            equation="max_growth_rate * (1 - population / carrying_capacity)",
        )
    )
    patch.upsert(Aux(name="max_growth_rate", equation="0.08"))
    patch.upsert(Aux(name="carrying_capacity", equation="10000"))
project.save_as(model_path)

m = simlin.open(model_path)
print(m.path.name, "revision", m.revision)

## Display the editor

The model as the last expression of a cell shows the Simlin diagram editor.
Drag a variable, add one from the tool dial, click a variable and change its
equation: each edit is saved to `logistic-growth.stmx` as you make it.

In [ ]:
m

## Simulate

`m.run()` simulates the model as it is on disk right now, including whatever
you just did in the editor. The loop-dominance analysis says which feedback
loop drives the S-curve when.

In [ ]:
run = m.run()
print(f"final population: {run.results['population'].iloc[-1]:.0f}")
for period in run.dominant_periods:
    span = f"t=[{period.start_time:.0f}, {period.end_time:.0f}]"
    print(f"{span} dominated by {period.dominant_loops}")

Edit from Python and the editor above follows (a short "Updated from Python"
notice); `m.selection` is the tuple of variables selected in the editor.

In [ ]:
from dataclasses import replace

with m.edit() as (current, patch):
    patch.upsert(replace(current["carrying_capacity"], equation="12000"))

print("revision", m.revision, "selection", m.selection)